In [1]:
import numpy as np
import pandas as pd
import math

# Max-Minority

* L’objectif de cette métrique est de **mesurer la pureté** d’un nœud **en évaluant sa capacité à
isoler ou à écraser la classe minoritaire** de manière linéaire.
* Pour un nœud t contenant N individus, la pureté P(t) est définie par la proportion de la
classe majoritaire :
**P(t) = max c∈{0,1} nc / N** 
* Exemple : Si un nœud contient 90% de feuilles saines et 10% de feuilles malades, sa pureté
est P(t) = max(0.9, 0.1) = 0.9. Si le nœud est parfaitement mélangé (50/50), alors P(t) = 0.5

In [2]:
df = pd.read_csv("../data/data.csv", sep=";")
df.head()

,pct_rouille,rugosite,nervure,est_malade
0,0.0,733.206356,0.0,0.0
1,0.0,879.171051,0.0,0.0
2,0.0,987.782414,0.0,0.0
3,0.0,265.555601,0.0,0.0
4,0.0,432.487942,0.0,0.0


In [3]:
df.columns

Index(['pct_rouille', 'rugosite', 'nervure', 'est_malade'], dtype='object')

In [4]:
X_test_purete = df["pct_rouille"]
y_purete = df["est_malade"]

In [5]:
def trouver_meilleur_split(X_column, y):
    N = len(X_column)

    def purete(y):
        n = len(y)
        sain_parmiX = len(y[y==0])
        malade_parmiX = len(y[y==1])

        pourcentage_sain = sain_parmiX / n
        pourcentage_malade = malade_parmiX / n

        return max(pourcentage_sain, pourcentage_malade)
    
    X_index_sorted = X_column.sort_values(ascending=True).index
    X_sorted = X_column.loc[X_index_sorted]
    seuils = []
    for i in range(N-1):
        if X_sorted.iloc[i] != X_sorted.iloc[i+1]:
            milieu = (X_sorted.iloc[i] + X_sorted.iloc[i+1]) / 2
            seuils.append(float(milieu))

    best_split = {"valeur":-math.inf, "seuil": -1, "indice_gauche": None, "indice_droite":None}
            
    for milieu in seuils:
        G = X_index_sorted[X_sorted < milieu]
        D = X_index_sorted[X_sorted >= milieu]
        y_G = y.loc[G]
        y_D = y.loc[D]

        purete_G = purete(y_G)
        purete_D = purete(y_D)

        Psplit = len(G)/N * purete_G + len(D)/N * purete_D

        if best_split["valeur"] < Psplit:
            best_split["valeur"] = Psplit
            best_split["seuil"] = milieu
            best_split["indice_gauche"] = G
            best_split["indice_droite"] = D

    return best_split

In [6]:
trouver_meilleur_split(X_test_purete, y_purete)

{'valeur': 0.9776516495211067,
 'seuil': 0.0067669808798841,
 'indice_gauche': Index([   0,  679,  682,  685,  686,  687,  688,  689,  690,  691,
        ...
         681,  513,  422, 1810, 1067,  693,  526, 2171,  270,  675],
       dtype='int64', length=1157),
 'indice_droite': Index([1897, 2172,  317, 1004, 2546, 1827, 1788, 1963, 1569,  767,
        ...
        2291, 2495, 1279, 2199, 2722, 2737, 1881, 1948, 1859, 1837],
       dtype='int64', length=1662)}

# Arbre de décision

In [28]:
class Noeud:
    def __init__(self, feature=None, split=None, seuil=None, droite=None, gauche=None, prediction=None):
        self.feature = feature
        self.split = split
        self.seuil = seuil
        self.droite = droite
        self.gauche = gauche
        self.prediction = prediction

In [8]:
def trouver_meilleur_split_dataset(X, y):
    columns = X.columns
    best_split = {"feature":None, "split":-math.inf, "seuil": -1, "indice_gauche": None, "indice_droite":None}
    for col in columns:
        split = trouver_meilleur_split(X[col], y)
        if best_split["split"] <= split["valeur"]:
            best_split = {"feature": col, "split": split["valeur"], "seuil": split["seuil"], "indice_gauche": split["indice_gauche"], "indice_droite":split["indice_droite"]}

    return best_split

In [29]:
def build_tree(X, y, depth=0, max_depth=100):
    if depth >= max_depth or y.nunique() == 1:
        sain_parmiX = len(y[y==0])
        malade_parmiX = len(y[y==1])
        return Noeud(prediction=0 if sain_parmiX > malade_parmiX else 1)

    split = trouver_meilleur_split_dataset(X, y)
        
    X_gauche = X.loc[split.get("indice_gauche")]
    y_gauche = y.loc[split.get("indice_gauche")]
    
    X_droite = X.loc[split.get("indice_droite")]
    y_droite = y.loc[split.get("indice_droite")]

    gauche = build_tree(X_gauche, y_gauche, depth+1, max_depth)
    droite = build_tree(X_droite, y_droite, depth+1, max_depth)

    return Noeud(
        feature=split["feature"],
        split=split["split"],
        seuil=split["seuil"],
        droite=droite,
        gauche=gauche,
        prediction=None
    )

In [10]:
X = df.drop(columns=["est_malade"])
y = df["est_malade"]

In [30]:
tree = build_tree(X, y, 0, max_depth=10)

In [38]:
print(tree.droite.droite.droite.prediction, tree.gauche.gauche.gauche.gauche.gauche.prediction)

1 0
